# Семинар 10: продвинутые функции

In [ ]:
def pow_slow(base: int | float, deg: int):
    if base == 0:
        return 0  # а без этого получим DivisionByZeroError, если deg < 0 [1]

    if deg == 0:
        return 1
    elif deg < 0:
        deg *= -1
        base = 1 / base

    current = 1
    for _ in range(deg):
        current *= base
    return current

In [8]:
pow_slow(2, 10000)

1995063116880758384883742162683585083823496831886192454852008949852943883022194663191996168403619459789933112942320912427155649134941378111759378593209632395785573004679379452676524655126605989552055008691819331154250860846061810468550907486608962488809048989483800925394163325785062156830947390255691238806522509664387444104675987162698545322286853816169431577562964076283688076073222853509164147618395638145896946389941084096053626782106462142733339403652556564953060314268023496940033593431665145929777327966577560617258203140799419817960737824568376228003730288548725190083446458145465055792960141483392161573458813925709537976911927780082695773567444412306201875783632550272832378927071037380286639303142813324140162419567169057406141965434232463880124885614730520743199225961179625013099286024170834080760593232016126849228849625584131284406153673895148711425631511108974551420331382020293164095759646475601040584584156607204496286701651506192063100418642227590867090057460641785695191145605506

### Рекурсия

Пусть мы хотим написать функцию быстрого возведения числа в степень:

```
def pow(base: int | float, deg: int)
```

Известно, что:
```
pow(a, n)
    = pow(a, n // 2) ** 2, если n -- четное
    = a * pow(a, n // 2) ** 2, если n -- нечетное
```

In [9]:
def pow(base: int | float, deg: int) -> int | float:
    if base == 0:
        return 0  # а без этого получим DivisionByZeroError, если deg < 0 [1]

    if deg == 0:
        return 1  # без этого получим бесконечное выполнение
    elif deg < 0:
        deg *= -1
        base = 1 / base  # [1] вот тут

    if deg % 2 == 0:
        return pow(base, deg // 2) ** 2
    return base * pow(base, deg // 2) ** 2


In [12]:
pow(2, -4)

0.0625

Что тут произошло? Правильно, мы вызвали из функции саму себя, получается так называемая рекурсия.

С рекурсией стоит быть аккуратным: неаккуратное ее использование приведет к бесконечной работе программы, а также она выжрет у вас всю память компа или питон прибьет ее раньше.

Под капотом рекурсия работает как стэк вызовов. Каждый новый вызов помещается на его верхушку, после выполнения работы функции на этапе рекурсии этот вызов со стэка снимается. Проще всего это заметить если Сделать искусственный пример, где выдадим ошибку в какой-то момент времени.

In [13]:
def recursive(n: int):
    if n == 0:
        raise RuntimeError("Just checking call stack")

    recursive(n - 1)


In [14]:
recursive(6)

RuntimeError: Just checking call stack

### Проблема лимита рекурсии

Пусть мы хотим рекурсивно сумму для всех чисел от 1 до n (примечание: ДА, я знаю, что это можно просто сделать циклом, это игрушечный пример)

In [15]:
def recursive(n: int):
    if n == 1:
        return 1

    return n + recursive(n - 1)

In [16]:
recursive(3)

6

In [17]:
recursive(10000)  # ой, а че это мы такое поймали?

RecursionError: maximum recursion depth exceeded

In [18]:
import sys

sys.getrecursionlimit()  # по умолчанию в питоне очень маленький лимит на рекурсивные вызовы

3000

In [19]:
sys.setrecursionlimit(10 ** 6)  # надо быть аккуратным

recursive(10000)

50005000

### Проблема отсутствия мемоизации

Пусть мы хотим посчитать $n$-ое число Фибоначчи

$F_1 = F_2 = 1$

$F_n = F_{n - 1} + F_{n - 2}$

In [20]:
def fib(n: int) -> int:
    if n <= 2:
        return 1
    return fib(n - 1) + fib(n - 2)  # рекурсивно берем предыдущие числа Фибоначчи

А теперь давайте подумаем: сколько будет работать такой код?

In [ ]:
%%time
fib(8)

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 7.15 μs


21

In [22]:
%%time
fib(34)  # тут уже будет долго

CPU times: user 554 ms, sys: 6.82 ms, total: 561 ms
Wall time: 652 ms


5702887

In [23]:
%%time
fib(40)  # и чем дальше, тем хуже, а 40 -- это же еще не то чтобы много

CPU times: user 9.96 s, sys: 108 ms, total: 10.1 s
Wall time: 13.8 s


102334155

Проблема заключается в том, что у нас есть дубликаты вызовов. Поэтому питон делает много дополнительных действий. Чем дальше по номеру число Фибоначчи -- тем больше придется решать одинаковых абсолютно вызовов рекурсии. Решить это можно мемоизацией:

In [36]:
from collections import defaultdict


# SENTINEL = object()


fib_cache = dict()


def fib(n: int) -> int:
    if n in fib_cache:
        return fib_cache[n]

    if n <= 2:
        fib_cache[n] = 1
        return 1

    result = fib(n - 1) + fib(n - 2)
    fib_cache[n] = result
    return result

In [37]:
%%time
fib(1000)  # быстро? быстро!

CPU times: user 208 μs, sys: 97 μs, total: 305 μs
Wall time: 439 μs


43466557686937456435688527675040625802564660517371780402481729089536555417949051890403879840079255169295922593080322634775209689623239873322471161642996440906533187938298969649928516003704476137795166849228875

Очевидно, что это уже получается громоздко и неудобно, поэтому кэширование в питоне есть в том числе и встроенное.

Есть так называемые декораторы `@cache` и `@lru_cache` (least recently used). Второй выкидывает из кэша элементы, когда его размер достиг заданного максимума. При этом `@cache` устроен вот так:

```(python)
def cache(user_function, /):
    return lru_cache(maxsize=None)(user_function)
```

У `@lru_cache` параметр maxsize по умолчанию равен 128, можно увеличить при желании.

In [ ]:
from functools import lru_cache, cache

# lru_cache -- можно еще указать параметр maxsize
@lru_cache(maxsize=10)  # <--- это декоратор, про них поговорим чуть позже
def fib(n: int) -> int:
    if n <= 2:
        return 1

    return fib(n - 1) + fib(n - 2)

In [53]:
%%time
fib(1000)

CPU times: user 213 μs, sys: 43 μs, total: 256 μs
Wall time: 256 μs


43466557686937456435688527675040625802564660517371780402481729089536555417949051890403879840079255169295922593080322634775209689623239873322471161642996440906533187938298969649928516003704476137795166849228875

In [64]:
def my_decorator(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        print("called", func.__name__, count, "times")
        return func(*args, **kwargs)
    return wrapper

@my_decorator
@lru_cache(maxsize=3)  # <--- это декоратор, про них поговорим чуть позже
def fib(n: int) -> int:
    if n <= 2:
        return 1

    return fib(n - 1) + fib(n - 2)

In [65]:
%%time
fib(1000)

called fib 1 times
called fib 2 times
called fib 3 times
called fib 4 times
called fib 5 times
called fib 6 times
called fib 7 times
called fib 8 times
called fib 9 times
called fib 10 times
called fib 11 times
called fib 12 times
called fib 13 times
called fib 14 times
called fib 15 times
called fib 16 times
called fib 17 times
called fib 18 times
called fib 19 times
called fib 20 times
called fib 21 times
called fib 22 times
called fib 23 times
called fib 24 times
called fib 25 times
called fib 26 times
called fib 27 times
called fib 28 times
called fib 29 times
called fib 30 times
called fib 31 times
called fib 32 times
called fib 33 times
called fib 34 times
called fib 35 times
called fib 36 times
called fib 37 times
called fib 38 times
called fib 39 times
called fib 40 times
called fib 41 times
called fib 42 times
called fib 43 times
called fib 44 times
called fib 45 times
called fib 46 times
called fib 47 times
called fib 48 times
called fib 49 times
called fib 50 times
called fi

43466557686937456435688527675040625802564660517371780402481729089536555417949051890403879840079255169295922593080322634775209689623239873322471161642996440906533187938298969649928516003704476137795166849228875